# 07 Figures for progress update 2 (Oct 21 – Nov 4)

This notebook reproduces every chart and table in the second progress-update deck. Each section reruns the experiment behind one slide with the repo's own code (`silent_rag.run`, the detectors in `src/silent_rag/detect`, the scoring in `src/silent_rag/eval`), draws the plot, and prints the notebook's number next to the number on the slide.

Unlike notebooks 01–06, this one is **not** self-contained: the point is to show that the slides come from the benchmark code. Run it from the repo's virtual environment with the package installed (`pip install -e .`). It uses **NFCorpus only**, so nothing here touches the held-out sets.

**Run time.** The two runner calls (sections 1 and 5) take about 30 and 25 minutes on a laptop CPU, almost all of it in the re-embedding check. Their results are saved to `results/progress2_*.parquet` (git-ignored) and loaded on the next run, so after the first run the notebook takes a minute or two. Delete those files to rerun from scratch. Embeddings are cached in `cache/embeddings` as usual.

| Slide | What it shows | Section |
|---|---|---|
| 5 | Search quality lost by a half-finished model upgrade | 1, 2 |
| 6 | Share of windows counted as harmful, with and without the 0.02 line | 3 |
| 7 | How well each monitor separates damaged from clean windows (AUROC) | 4 |
| 8 | Time per check | 5 |
| 9 | PSI on two samples of the same data, against the 0.25 rule | 6 |
| 10 | Alarm rates on harmless changes | 7 |
| 11 | Effect of re-chunking every document | 8 |

The section numbers follow the order of the slides. Small differences from the slides are expected where the slide was taken from an earlier run with another seed or trial count. The last cell lists every comparison in one table.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # On macOS, this prevents an OpenMP conflict. It has no effect on other systems.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import copy
import dataclasses
import hashlib
import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# run from the repo root, so configs/ and results/ resolve the same way they do for the runner
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT)

from silent_rag import data
from silent_rag import run as R
from silent_rag.detect.psi import PSI_DETECTORS
from silent_rag.eval.detector_scoring import compute_auroc
from silent_rag.eval.ground_truth import paired_bootstrap, per_query_ndcg
from silent_rag.inject.base import REGISTRY as INJECTORS
from silent_rag.inject.base import Injector, register
from silent_rag.pipeline import Retriever
from silent_rag.types import Observation, RetrievalState

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e5e5e5", "axes.axisbelow": True})
TEAL, BRICK, AMBER, GREY = "#2F6F7E", "#B8452F", "#E39B2D", "#C9CED6"

DEV = yaml.safe_load(open("configs/dev.yaml"))
MINILM, BGE = "F1_partial_upgrade_minilm", "F1_partial_upgrade_bge"
SEVERITIES = [0.1, 0.5, 0.9]

# what the deck shows, for the comparison table at the end
compare = []


def check(slide, what, on_slide, here, fmt="{:.3f}"):
    here = here.item() if hasattr(here, "item") else here
    compare.append({"slide": slide, "number": what, "on slide": on_slide, "this notebook": fmt.format(here)})

## Running the benchmark

`cached_run` runs a config through `silent_rag.run.run`, the same function `python -m silent_rag.run` calls, and saves the result table. The file name includes a hash of the config, so a changed config never reads an old result.

Each row of the result is one detector on one window of 100 queries: the detector's `score`, the window's `ndcg_drop` against clean retrieval, the bootstrap `p_value`, the `harmful` label, and `cost_s`, the time the detector took to score that window.

In [ ]:
def cached_run(name, cfg):
    key = hashlib.sha1(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:8]
    path = Path(f"results/progress2_{name}_{key}.parquet")
    if path.exists():
        print(f"loading {path}")
        return pd.read_parquet(path)
    start = time.time()
    df = R.run(copy.deepcopy(cfg))
    path.parent.mkdir(exist_ok=True)
    df.to_parquet(path, index=False)
    print(f"wrote {len(df)} rows to {path} in {(time.time() - start) / 60:.0f} min")
    return df


def windows(df):
    # every detector sees the same windows, so the ground truth is one row per window
    return df.drop_duplicates(["failure", "severity", "seed", "trial"])

## 1. The half-finished upgrade (slides 4–8)

The failure from `configs/dev.yaml`: a share of the documents (10%, 50% or 90%) is re-embedded with a different model, while questions are still embedded with the original one. The new model is either `all-MiniLM-L6-v2` or `bge-small-en-v1.5`.

This is `configs/dev.yaml` with every registered monitor attached, so one run feeds slides 5 to 8. The monitor names on the slides map to the detectors like this:

| Slide | Detector |
|---|---|
| Re-check vectors | `re_embed` |
| Top-score shift | `ks_top1` |
| Average top score | `ks_mean10` |
| PSI | `psi_top1` |
| Gap | `ks_gap` |
| Document classifier | `domain_classifier_retrieved_docs` |
| Question classifier | `domain_classifier_query` |

In [ ]:
LABELS = {
    "re_embed": "Re-check vectors",
    "ks_top1": "Top-score shift",
    "ks_mean10": "Average top score",
    "psi_top1": "PSI",
    "ks_gap": "Gap",
    "domain_classifier_retrieved_docs": "Document classifier",
    "domain_classifier_query": "Question classifier",
}
main_cfg = {**DEV, "detectors": ["ks_top1", "ks_gap", "ks_mean10", "ks_std10", "psi_top1", "psi_gap", "psi_mean10", "psi_std10",
                                 "re_embed", "domain_classifier_query", "domain_classifier_retrieved_docs"]}
main = cached_run("f1_all_detectors", main_cfg)
main.groupby("detector").size()

## 2. How much search gets worse (slide 5)

The mean nDCG@10 drop over the windows at each severity, all three seeds together (60 windows per bar). The table also shows seed 0 alone, to show how close one seed is to the mean.

In [ ]:
w = windows(main)
damage = w[w.failure != "clean"].pivot_table(index="severity", columns="failure", values="ndcg_drop")
damage_seed0 = w[(w.failure != "clean") & (w.seed == 0)].pivot_table(index="severity", columns="failure", values="ndcg_drop")
display(pd.concat({"seed 0": damage_seed0, "3 seeds": damage}, axis=1).round(3))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
x = np.arange(3)
for off, (f, col, lab) in zip((-0.18, 0.18), [(MINILM, TEAL, "Upgrade to all-MiniLM"), (BGE, BRICK, "Upgrade to bge-small")]):
    ax.bar(x + off, damage[f], 0.34, color=col, label=lab)
    for xi, v in zip(x, damage[f]):
        ax.text(xi + off, v + 0.003, f"{v:.3f}", ha="center", fontsize=8)
ax.axhline(0.02, color="#4A5566", ls="--", lw=0.8)
ax.text(2.45, 0.022, "0.02", fontsize=8, color="#4A5566")
ax.set_xticks(x, [f"{s:.0%} of documents" for s in SEVERITIES])
ax.set_ylabel("Drop in nDCG@10")
ax.set_title("Search quality lost by a half-finished upgrade", loc="left")
ax.legend(frameon=False)
plt.show()

for f, vals in [(MINILM, [0.009, 0.044, 0.065]), (BGE, [0.014, 0.080, 0.195])]:
    for s, v in zip(SEVERITIES, vals):
        check(5, f"drop, {f.split('_')[-1]} at {s:.0%}", v, damage.loc[s, f])

## 3. What counts as harmful (slide 6)

A window is harmful when its nDCG@10 drop is significant (`p < 0.05`, paired bootstrap). The runner's default `min_drop` is 0, so any significant drop counts. With `min_drop: 0.02` a window also has to lose at least 0.02. The labels are recomputed from `p_value` and `ndcg_drop` here, so both lines come from the same run. MiniLM only, all three seeds (60 windows per bar).

In [ ]:
m = w[w.failure == MINILM]
share = pd.DataFrame({
    "any significant drop": m.groupby("severity").apply(lambda g: (g.p_value < 0.05).mean()),
    "drop >= 0.02": m.groupby("severity").apply(lambda g: ((g.p_value < 0.05) & (g.ndcg_drop >= 0.02)).mean()),
}) * 100
display(share.round(0))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
for off, (col, c) in zip((-0.18, 0.18), [("any significant drop", GREY), ("drop >= 0.02", TEAL)]):
    ax.bar(x + off, share[col], 0.34, color=c, label=col)
    for xi, v in zip(x, share[col]):
        ax.text(xi + off, v + 2, f"{v:.0f}%", ha="center", fontsize=8)
ax.set_xticks(x, [f"{s:.0%} of documents" for s in SEVERITIES])
ax.set_ylim(0, 112)
ax.set_ylabel("% of windows harmful")
ax.set_title("Upgrade to all-MiniLM: harmful windows", loc="left")
ax.legend(frameon=False, loc="lower right")
plt.show()

for s, a, b in zip(SEVERITIES, [55, 100, 100], [3, 100, 100]):
    check(6, f"harmful, any drop, {s:.0%}", a, share.loc[s, "any significant drop"], "{:.0f}")
    check(6, f"harmful, drop >= 0.02, {s:.0%}", b, share.loc[s, "drop >= 0.02"], "{:.0f}")

## 4. Which monitors notice (slide 7)

AUROC from `compute_auroc`: the harmful windows of the failure against the clean windows, with the runner's labels (`min_drop` 0). 1 means the monitor always scores a damaged window above a clean one, 0.5 is a coin flip. The slide shows the MiniLM upgrade, all three seeds together. The table also shows the bge upgrade.

In [ ]:
auroc = pd.DataFrame({
    f: {LABELS.get(d, d): compute_auroc(main, d, f) for d in sorted(main.detector.unique())} for f in (MINILM, BGE)
}).sort_values(MINILM, ascending=False)
display(auroc.round(2))

shown = [LABELS[d] for d in ["re_embed", "ks_top1", "ks_mean10", "psi_top1", "ks_gap"]]
vals = auroc.loc[shown, MINILM]
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.barh(shown[::-1], vals[::-1], color=[AMBER if n == "Top-score shift" else GREY for n in shown[::-1]])
for y, v in enumerate(vals[::-1]):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=8)
ax.axvline(0.5, color="#4A5566", ls="--", lw=0.8)
ax.set_xlim(0, 1.1)
ax.set_xlabel("AUROC, upgrade to all-MiniLM")
ax.set_title("How well each monitor tells damaged from clean", loc="left")
ax.grid(axis="y", visible=False)
plt.show()

for name, v in zip(shown, [1.00, 0.91, 0.90, 0.84, 0.49]):
    check(7, f"AUROC, {name}", v, auroc.loc[name, MINILM], "{:.2f}")

## 5. What each check costs (slide 8)

The median of `cost_s` per detector over every window of the run in section 1: the time to score one window of 100 queries, after the detector has been fit. Timings depend on the machine and on what else it is running, so expect different absolute numbers from the slide. The ratios are the point: the re-embedding check costs thousands of times as much as a KS test. Log scale.

In [ ]:
cost = main.groupby("detector").cost_s.median().sort_values() * 1000
display(cost.rename("median ms per window").round(2).to_frame())

shown = ["ks_top1", "domain_classifier_query", "domain_classifier_retrieved_docs", "re_embed"]
fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.barh([LABELS[d] for d in shown], cost[shown], color=[GREY, GREY, GREY, BRICK])
for y, d in enumerate(shown):
    v = cost[d]
    ax.text(v * 1.15, y, f"{v / 1000:.1f} s" if v >= 1000 else f"{v:.1f} ms" if v < 10 else f"{v:.0f} ms", va="center", fontsize=8)
ax.set_xscale("log")
ax.set_xlim(0.5, 3e4)
ax.set_xlabel("Median time per window (ms, log scale)")
ax.set_title("Time to check one window of 100 questions", loc="left")
ax.grid(axis="y", visible=False)
plt.show()

for d, v in zip(shown, [1.7, 23, 119, 3700]):
    check(8, f"ms per window, {LABELS[d]}", v, cost[d], "{:.1f}")

## 6. The 0.25 PSI rule on clean data (slide 9)

A common rule of thumb says a PSI above 0.25 means the data has shifted. Here both samples come from the same distribution, so every alarm is false. Each of the 4,000 simulations fits the repo's `psi_top1` detector on 100 values and scores another 100 from the same normal distribution. That is the window size of the benchmark. The score lists are made up, and only the top-1 column matters to `psi_top1`.

In [ ]:
def fake_window(top1):
    # 10 scores per query, sorted high to low, top-1 first; the other nine only make the shape valid
    scores = np.hstack([top1[:, None], top1[:, None] - np.linspace(0.01, 0.1, 9)])
    n = len(top1)
    return Observation(query_ids=[str(i) for i in range(n)], query_embeddings=np.zeros((n, 2)),
                       topk_ids=[[]] * n, topk_scores=scores)


rng = np.random.default_rng(0)
null = []
for _ in range(4000):
    det = PSI_DETECTORS["top1"]()
    det.fit(fake_window(rng.normal(0, 1, 100)))
    null.append(det.score(fake_window(rng.normal(0, 1, 100))))
null = np.array(null)
print(f"mean PSI {null.mean():.3f}, above 0.25 in {(null > 0.25).mean():.0%} of simulations")

edges = np.arange(0, 0.65, 0.05)
counts = [int(c) for c in np.histogram(null, bins=edges)[0]] + [int((null >= 0.6).sum())]
labels = [f"{a:.2f}" for a in edges[:-1]] + ["0.60+"]
fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.bar(labels, counts, color=[BRICK if a >= 0.25 else GREY for a in list(edges[:-1]) + [0.6]], width=0.85)
ax.set_xlabel("PSI between two samples of the same data (bin start)")
ax.set_ylabel("Simulations")
ax.set_title("A common alarm rule fires on nothing", loc="left")
ax.tick_params(axis="x", labelsize=7)
plt.show()

print("bin counts:", counts)
check(9, "share of simulations above 0.25", "22%", (null > 0.25).mean() * 100, "{:.0f}%")
check(9, "bin counts", "78, 536, 968, 875, 676, 410, 238, 91, 52, 29, 11, 5, 31", ", ".join(map(str, counts)), "{}")

## 7. Harmless changes (slide 10)

Changes that alter the index without making search worse should not set off an alarm. These two injectors were written for #9 and are defined here because they are not in `src/` yet:

- `H_mixed_truncation`: a share of the documents is re-embedded with the same model at `max_seq_length` 512 instead of 256. The slide shows the 90% condition.
- `H_full_swap`: documents **and** questions move to another model. The system is consistent, just different.

The alarm line for each monitor is the highest score it gave any clean window, over all three seeds. The table is the % of windows of each change that score above that line.

In [ ]:
def like(retriever, **changes):
    kw = dict(normalize=retriever.normalize, query_prefix=retriever.query_prefix, max_seq_length=retriever.max_seq_length,
              revision=retriever.revision, batch_size=retriever.batch_size, device=retriever.device)
    return Retriever(changes.pop("model", retriever.model), **{**kw, **changes})


# re-running this cell would register a second class under the same names
for name in ("H_mixed_truncation", "H_full_swap"):
    INJECTORS.pop(name, None)


@register
class MixedTruncation(Injector):
    # A `severity` share of the documents re-embedded at another max_seq_length, same model.
    name = "H_mixed_truncation"
    harmful = False

    def __init__(self, max_seq_length: int):
        self.alt_len = max_seq_length

    def attach(self, retriever, state):
        self.alt, _ = like(retriever, max_seq_length=self.alt_len).embed_texts("nfcorpus", state.doc_ids, state.doc_texts)

    def apply(self, state, severity, seed):
        n = len(state.doc_ids)
        rows = np.random.default_rng(seed).permutation(n)[: round(severity * n)]
        emb = state.doc_embeddings.copy()
        emb[rows] = self.alt[rows]
        return dataclasses.replace(state, doc_embeddings=emb)


@register
class FullSwap(Injector):
    # Documents and questions both on another model. All or nothing, so run it at severity 1.0.
    name = "H_full_swap"
    harmful = False

    def __init__(self, alt_model: str, alt_revision: str):
        self.alt_model, self.alt_revision = alt_model, alt_revision

    def attach(self, retriever, state):
        alt = like(retriever, model=self.alt_model, revision=self.alt_revision)
        self.docs, _ = alt.embed_texts("f1_alt", state.doc_ids, state.doc_texts)  # the vectors F1 already cached
        self.queries = alt.embed_queries(state.query_texts)

    def apply(self, state, severity, seed):
        return dataclasses.replace(state, doc_embeddings=self.docs, query_embeddings=self.queries)


f1 = DEV["failures"]
controls_cfg = {
    **DEV,
    "failures": {
        "H_trunc512": {"injector": "H_mixed_truncation", "severities": SEVERITIES, "max_seq_length": 512},
        "H_swap_minilm": {"injector": "H_full_swap", "severities": [1.0],
                          "alt_model": f1[MINILM]["alt_model"], "alt_revision": f1[MINILM]["alt_revision"]},
        "H_swap_bge": {"injector": "H_full_swap", "severities": [1.0],
                       "alt_model": f1[BGE]["alt_model"], "alt_revision": f1[BGE]["alt_revision"]},
    },
    "detectors": ["ks_top1", "re_embed", "domain_classifier_query", "domain_classifier_retrieved_docs"],
}
controls = cached_run("harmless_changes", controls_cfg)

In [ ]:
cw = windows(controls)
print("these changes really are harmless, mean nDCG@10 drop:")
display(cw[cw.failure != "clean"].groupby(["failure", "severity"]).ndcg_drop.mean().round(4).to_frame())

cols = {("H_trunc512", 0.9): "Longer documents (512), 90%", ("H_swap_minilm", 1.0): "Everything switched to MiniLM",
        ("H_swap_bge", 1.0): "Everything switched to bge"}
alarms = {}
for d, g in controls.groupby("detector"):
    line = g[g.failure == "clean"].score.max()
    alarms[LABELS[d]] = {lab: (g[(g.failure == f) & (g.severity == s)].score > line).mean() * 100 for (f, s), lab in cols.items()}
alarms = pd.DataFrame(alarms).T.loc[["Top-score shift", "Re-check vectors", "Document classifier", "Question classifier"]]
display(alarms.round(0).astype(int).style.set_caption("% of windows that raise an alarm"))

on_slide = {"Top-score shift": [8, 48, 100], "Re-check vectors": [100, 100, 100],
            "Document classifier": [100, 100, 100], "Question classifier": [0, 100, 100]}
for mon, vals in on_slide.items():
    for col, v in zip(alarms.columns, vals):
        check(10, f"alarms %, {mon}, {col}", v, alarms.loc[mon, col], "{:.0f}")

## 8. Re-chunking every document (slide 11)

The planned chunking failure, measured before building it. Every NFCorpus document is split into token windows of the given size (and overlap), each chunk is embedded with the clean model, and a document ranks by its best chunk. The baseline is the one vector per document, cut at 256 tokens, that the benchmark uses. All 323 test queries, change in mean nDCG@10 (positive means better).

The chunk embeddings take a few minutes the first time and are cached under `f3check_*`.

In [ ]:
from transformers import AutoTokenizer

r = Retriever(DEV["model"], **DEV["retriever"])
tok = AutoTokenizer.from_pretrained(DEV["model"], revision=DEV["retriever"]["revision"])
ds = data.load("nfcorpus")
clean = r.embed_corpus(ds)
ids, _, _ = r.search(clean, k=10, query_embeddings=clean.query_embeddings)
base = per_query_ndcg(ids, clean.query_ids, ds.qrels, 10)
body = tok(clean.doc_texts, add_special_tokens=False)["input_ids"]

# 252 + [CLS] + [SEP] = 254 <= max_seq_length 256, so no chunk is truncated
SETTINGS = {"252 tokens": (252, 0), "252, overlap 63": (252, 63), "126 tokens": (126, 0), "63 tokens": (63, 0)}
rows = []
for label, (size, overlap) in SETTINGS.items():
    texts, parents = [], []
    for d, toks in zip(clean.doc_ids, body):
        for s in range(0, max(len(toks) - overlap, 1), size - overlap):
            texts.append(tok.decode(toks[s:s + size]))
            parents.append(d)
    vecs, _ = r.embed_texts(f"f3check_{size}_{overlap}", [f"{p}#{i}" for i, p in enumerate(parents)], texts)
    st = RetrievalState(doc_ids=parents, doc_texts=[""] * len(parents), doc_embeddings=vecs, query_ids=clean.query_ids,
                        query_texts=clean.query_texts, query_embeddings=clean.query_embeddings, parent_ids=parents)
    cids, _, _ = r.search(st, k=10, query_embeddings=clean.query_embeddings)
    nd = per_query_ndcg(cids, clean.query_ids, ds.qrels, 10)
    drop, p = paired_bootstrap(base, nd, seed=0)
    rows.append({"chunking": label, "chunks": len(texts), "nDCG@10": nd.mean(), "change": nd.mean() - base.mean(), "p (drop)": p})
chunking = pd.DataFrame(rows).set_index("chunking")
print(f"baseline nDCG@10 {base.mean():.4f}")
display(chunking.round(4))

fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.bar(chunking.index, chunking.change, color=[GREY if v < 0 else TEAL for v in chunking.change], width=0.55)
for i, v in enumerate(chunking.change):
    ax.text(i, v + (0.0006 if v >= 0 else -0.0014), f"{v:+.3f}", ha="center", fontsize=8)
for y in (0.02, -0.02):
    ax.axhline(y, color="#4A5566", ls="--", lw=0.8)
ax.axhline(0, color="#4A5566", lw=0.8)
ax.set_ylim(-0.025, 0.025)
ax.set_ylabel("Change in nDCG@10")
ax.set_title("Re-chunking every document barely moves search", loc="left")
plt.show()

for label, v in zip(SETTINGS, [-0.004, -0.002, 0.004, 0.010]):
    check(11, f"nDCG change, {label}", v, chunking.loc[label, "change"])

## Notebook vs slides

Every number this notebook checked, next to the number on the slide.

In [ ]:
pd.DataFrame(compare)